<a href="https://colab.research.google.com/github/RenatGreen-flag/Model-Liniar-Regression/blob/main/goldPricingPrediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [62]:
import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime

TROY_OUNCE_TO_GRAM = 31.1035   # 1 troy ounce = 31.1035 gram

ticker_map = {
    "GC=F"    : "gold",
    "CL=F"    : "oil",
    "^JKSE"   : "ihsg",
    "^GSPC"   : "snp500",
    "^TNX"    : "treasury_us",   # Hapus tanda '-' agar aman sebagai nama kolom
    "USDIDR=X": "usdidr",
}

START = "2023-01-01"
END   = datetime.today().strftime("%Y-%m-%d")

LAG_STEPS       = [1, 3, 7]
ROLLING_WINDOWS = [3, 7]

In [63]:
print("=" *55)
print("TAHAP 1: Mengunduh data dari Yahoo Finance...")
print("=" * 55)

raw = yf.download(
    tickers     = list(ticker_map.keys()),
    start       = START,
    end         = END,
    auto_adjust = True,
    progress    = True
)

df = raw["Close"].rename(columns=ticker_map).add_suffix("_close")
print(f"\nShape setelah download: {df.shape}")
print(f"Rentang tanggal awal  : {df.index.min().date()} s.d. {df.index.max().date()}")


[*********************100%***********************]  6 of 6 completed

TAHAP 1: Mengunduh data dari Yahoo Finance...

Shape setelah download: (900, 6)
Rentang tanggal awal  : 2023-01-02 s.d. 2026-06-18


In [64]:
# TAHAP 2: PENYELARASAN TIMEZONE (SHIFT +1 HARI)
GLOBAL_TICKERS = list(ticker_map.keys())

for ticker_name in GLOBAL_TICKERS:
  col = f"{ticker_name}_close"
  if col in df.columns:
    df[col] = df[col].shift(1)


In [65]:
# TAHAP 3: REINDEX KE KALENDER HARIAN PENUH

full_range = pd.date_range(start=df.index.min(), end=df.index.max(), freq="D")
df = df.reindex(full_range)
df.index.name = "Date"

print(f"Shape telah reindex: {df.shape}")


Shape telah reindex: (1264, 6)


In [66]:
# TAHAP 4: FORWARD FILL (FFILL) -> HARGA CLOSE HARI TERAKHIR

price_cols = [c for c in df.columns if c.endswith("_close")]
df[price_cols] = df[price_cols].ffill().bfill()

missing_after = df[price_cols].isnull().sum()
print(f"\nJumlah missing value setelah forward fill:\n{missing_after}")



Jumlah missing value setelah forward fill:
Ticker
oil_close            0
gold_close           0
usdidr_close         0
snp500_close         0
ihsg_close           0
treasury_us_close    0
dtype: int64


In [67]:
# # feature engineering gold usd -> idr per gram
df["gold_close_idr_per_oz"] = df["gold_close"] * df['usdidr_close']

df["gold_close_idr_per_gram"] = df["gold_close_idr_per_oz"] / TROY_OUNCE_TO_GRAM

price_cols = price_cols + ["gold_close_idr_per_oz","gold_close_idr_per_gram"]
print(df[["gold_close", "usdidr_close", "gold_close_idr_per_oz", "gold_close_idr_per_gram"]].tail(3).to_string())

Ticker       gold_close  usdidr_close  gold_close_idr_per_oz  gold_close_idr_per_gram
Date                                                                                 
2026-06-16  4330.899902       17716.0           7.672622e+07             2.466804e+06
2026-06-17  4358.899902       17730.0           7.728330e+07             2.484714e+06
2026-06-18  4224.100098       17891.0           7.557337e+07             2.429739e+06


In [68]:
# # feature engineering : lag feature (H-1, H-3, H-7)

for col in price_cols:
  for lag in LAG_STEPS:
    df[f"{col}_lag{lag}"] = df[col].shift(lag)

lag_cols = [c for c in df.columns if "gold" in c and "lag" in c]
print(f"Contoh kolom lag gold: {lag_cols}")

Contoh kolom lag gold: ['gold_close_lag1', 'gold_close_lag3', 'gold_close_lag7', 'gold_close_idr_per_oz_lag1', 'gold_close_idr_per_oz_lag3', 'gold_close_idr_per_oz_lag7', 'gold_close_idr_per_gram_lag1', 'gold_close_idr_per_gram_lag3', 'gold_close_idr_per_gram_lag7']


In [69]:
# # feature engineering: Rolling statistic per time step

for col in price_cols:
  for window in ROLLING_WINDOWS:
    df[f"{col}_rmean_{window}"] = df[col].rolling(window=window).mean()
    df[f"{col}_rstd_{window}"] = df[col].rolling(window=window).std()
    df[f"{col}_rmin_{window}"] = df[col].rolling(window=window).min()
    df[f"{col}_rmax_{window}"] = df[col].rolling(window=window).max()

In [70]:
# # Featur engineering: persentase perubahan (RETURN HARIAN)
for col in price_cols:
  df[f"{col}_pct_change"] = df[col].pct_change()

df_clean = df.dropna().reset_index()
print(f"\nShape FINAL setelah semua tahap preprocessing: {df_clean.shape}")
print(f"Rentang tanggal final : {df_clean['Date'].min().date()} s.d. {df_clean['Date'].max().date()}")

/tmp/ipykernel_1971/1473989071.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_pct_change"] = df[col].pct_change()
/tmp/ipykernel_1971/1473989071.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df[f"{col}_pct_change"] = df[col].pct_change()
/tmp/ipykernel_1971/1473989071.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragment


Shape FINAL setelah semua tahap preprocessing: (1257, 105)
Rentang tanggal final : 2023-01-09 s.d. 2026-06-18


In [71]:
# # TAHAP 10: SIMPAN KE CSV + RINGKASAN DATASET

output_path = "gold_preprocessed_final.csv"
df_clean.to_csv(output_path, index=False)


In [72]:
# print("\n" + "=" * 55)
# print("RINGKASAN DATASET FINAL")
# print("=" * 55)

# # Pengelompokan kolom berdasarkan tipe fitur
# cols_raw     = [c for c in df_clean.columns if c.endswith("_close") and "lag" not in c and "rmean" not in c]
# cols_derived = ["gold_idr_per_oz", "gold_idr_per_gram"]
# cols_lag     = [c for c in df_clean.columns if "_lag" in c]
# cols_rolling = [c for c in df_clean.columns if any(x in c for x in ["_rmean_", "_rstd_", "_rmin_", "_rmax_"])]
# cols_return  = [c for c in df_clean.columns if "_pct_change" in c]

# print(f"\n{'Tipe Fitur':<25} {'Jumlah Kolom':>13}")
# print("-" * 40)
# print(f"{'Kolom mentah (_close)':<25} {len(cols_raw):>13}")
# print(f"{'Fitur turunan (IDR)':<25} {len(cols_derived):>13}")
# print(f"{'Fitur lag (H-1/3/7)':<25} {len(cols_lag):>13}")
# print(f"{'Rolling statistics':<25} {len(cols_rolling):>13}")
# print(f"{'Return harian':<25} {len(cols_return):>13}")
# print("-" * 40)
# print(f"{'TOTAL KOLOM (+ Date)':<25} {df_clean.shape[1]:>13}")
# print(f"{'Total baris':<25} {df_clean.shape[0]:>13}")

# print(f"\nFile tersimpan ke: {output_path}")
# print("\n5 baris pertama dataset:")
# print(df_clean[["Date", "gold_close", "usdidr_close", "gold_close_idr_per_gram",
#                  "gold_close_lag1", "gold_close_lag7",
#                  "gold_close_rmean_7", "gold_close_pct_change"]].head().to_string(index=False))

In [73]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# Load the dataset
pd.set_option("display.max_rows", 120)
data_clean = pd.read_csv('/content/gold_preprocessed_final.csv', parse_dates=['Date'])
df = pd.DataFrame(data_clean)

df['Date'] = pd.to_datetime(data_clean['Date'])

df = df.sort_values('Date')

print(df.tail())

           Date  oil_close   gold_close  usdidr_close  snp500_close  \
1252 2026-06-14  84.879997  4215.000000       17921.0   7431.459961   
1253 2026-06-15  80.750000  4328.000000       17772.0   7554.290039   
1254 2026-06-16  76.050003  4330.899902       17716.0   7511.350098   
1255 2026-06-17  76.790001  4358.899902       17730.0   7420.100098   
1256 2026-06-18  76.599998  4224.100098       17891.0   7420.100098   

       ihsg_close  treasury_us_close  gold_close_idr_per_oz  \
1252  6007.655762              4.487           7.553702e+07   
1253  6254.965820              4.487           7.691722e+07   
1254  6254.965820              4.487           7.672622e+07   
1255  6220.740234              4.487           7.728330e+07   
1256  6220.740234              4.487           7.557337e+07   

      gold_close_idr_per_gram  oil_close_lag1  ...  \
1252             2.428570e+06       84.879997  ...   
1253             2.472944e+06       84.879997  ...   
1254             2.466804e+06   

In [74]:
print(f"Shape Awal: {df.shape}")

Shape Awal: (1257, 105)


In [75]:
print("TAHAP 1: EDA - Korelasi fitur terhadap gold_close")

corr_target = df.select_dtypes(include=[np.number]).corr()['gold_close_idr_per_gram']
corr_target_sorted = corr_target.drop('gold_close_idr_per_gram').sort_values(key=abs, ascending=False)

print(corr_target_sorted.head(10))
print(corr_target_sorted.tail(10))

low_variance = df.select_dtypes(include=[np.number]).std().sort_values()
print(low_variance.head(5))

TAHAP 1: EDA - Korelasi fitur terhadap gold_close
gold_close_idr_per_oz              1.000000
gold_close_idr_per_gram_rmean_3    0.999597
gold_close_idr_per_oz_rmean_3      0.999597
gold_close_idr_per_oz_rmin_3       0.999496
gold_close_idr_per_gram_rmin_3     0.999496
gold_close_idr_per_oz_rmax_3       0.999296
gold_close_idr_per_gram_rmax_3     0.999296
gold_close_idr_per_oz_lag1         0.999186
gold_close_idr_per_gram_lag1       0.999186
gold_close_idr_per_gram_rmean_7    0.998875
Name: gold_close_idr_per_gram, dtype: float64
ihsg_close_pct_change                -0.048872
usdidr_close_rstd_7                  -0.048440
usdidr_close_rstd_3                  -0.032351
gold_close_idr_per_oz_pct_change      0.030051
gold_close_idr_per_gram_pct_change    0.030051
oil_close_pct_change                  0.027661
usdidr_close_pct_change               0.024509
gold_close_pct_change                 0.020722
treasury_us_close_pct_change         -0.010888
snp500_close_pct_change              -0.0

In [76]:
base_cols = [c for c in df.columns if c.endswith("_close") and "lag" not in c and "rmean" not in c
             and "rmean" not in c and 'rstd' not in c and 'rmin' not in c and 'rmax' not in c]

plt.figure(figsize=(9,7))
sns.heatmap(df[base_cols].corr(), annot=True, fmt="2f", cmap='RdYlGn', center=0, vmin=-1, vmax=1)
plt.title('Korelasi Antar Variabel Dasar (sebelum lag/rolling)')
plt.tight_layout()
plt.savefig('eda_heatmap_korelasi.png', dpi=150)
plt.close()

In [77]:
# import pandas as pd
# import seaborn as sns
# import matplotlib.pyplot as plt

# df_60_days = df.tail(30)

# plt.figure(figsize=(12, 6))
# plt.plot(df_60_days['Date'], df_60_days['gold_close_idr_per_gram'])

# plt.title('Pergerakan Harga Emas (60 Hari Terakhir)')
# plt.xlabel('Tanggal')
# plt.ylabel('Harga Emas (IDR/Gram)')
# plt.grid(True)
# plt.xticks(rotation=45)

# plt.tight_layout()
# plt.show()